# Document Downloader - Municipalidad de Rosario

This notebook downloads the documents listed in the open-data CSVs and saves them directly to your Google Drive.

**Steps:**
1. Run the Drive mount cell
2. Upload the scrapper CSVs to the configured folder
3. Run the remaining cells

> **Tip:** To prevent the session from expiring on free Colab, connect with Colab Pro or use the Chrome anti-idle extension.

In [ ]:
# ── 1. Mount Google Drive ───────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# Set your base folder in Drive
DRIVE_BASE = '/content/drive/MyDrive/Rosario_Docs'
SCRAPPER_DIR = '/content/drive/MyDrive/Rosario_Docs/Scrapper'  # where you uploaded the CSVs

import os
os.makedirs(DRIVE_BASE, exist_ok=True)
print('Drive mounted OK. Base folder:', DRIVE_BASE)

In [ ]:
# ── 2. Install dependencies ─────────────────────────────────────────
!pip install -q aiofiles aiohttp tqdm beautifulsoup4 lxml

In [ ]:
# ── 3. Download the script from GitHub (or upload it manually) ──────
# Option A: from GitHub (if you pushed the repo)
# !wget -q https://raw.githubusercontent.com/YOUR_USER/YOUR_REPO/main/downloader.py

# Option B: paste the script directly (generates the file)
script_code = '''
# (paste the full content of downloader.py here)
'''

# Option C: if you already uploaded downloader.py to Drive
import shutil
shutil.copy(f'{DRIVE_BASE}/downloader.py', '/content/downloader.py')
print('Script copied OK')

In [ ]:
# ── 4. Configure paths and run the downloader ───────────────────────
import subprocess, sys

OUTPUT_DIR   = f'{DRIVE_BASE}/downloads'
CONCURRENCY  = 6    # simultaneous downloads (lower if rate-limited)
DELAY        = 0.5  # seconds between requests

# Override SCRAPPER_DIR inside the script with an inline patch
# (simpler: pass environment variables)
import os
env = os.environ.copy()
env['SCRAPPER_DIR'] = SCRAPPER_DIR

cmd = [
    sys.executable, '/content/downloader.py',
    '--output', OUTPUT_DIR,
    '--concurrency', str(CONCURRENCY),
    '--delay', str(DELAY),
]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, env=env)

In [ ]:
# ── 5. Downloaded files summary ──────────────────────────────────────
from pathlib import Path

base = Path(OUTPUT_DIR)
for folder in sorted(base.iterdir()):
    if folder.is_dir():
        count = len(list(folder.glob('*.pdf')))
        size_mb = sum(f.stat().st_size for f in folder.glob('*.pdf')) / 1_048_576
        print(f'{folder.name:45s} {count:5d} files  {size_mb:8.1f} MB')